In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

## Load data & check for problems

In [ ]:
df = pd.read_csv("ab_data.csv")
df.head()

In [ ]:
print(df.shape)
print(df.isnull().sum())

Control users should only ever see `old_page`, and treatment users only `new_page`.
Rows where that's not true are unreliable — we don't know which page actually caused
the recorded result, so they have to go.

In [ ]:
mismatch = df[((df.group == 'control') & (df.landing_page == 'new_page')) |
              ((df.group == 'treatment') & (df.landing_page == 'old_page'))]
print("mismatched rows:", len(mismatch))

df_clean = df.drop(mismatch.index)

Each user should only appear once in the dataset. Check for leftover duplicates
after removing the mismatches above.

In [ ]:
print("duplicates left:", df_clean.duplicated(subset='user_id').sum())

df_clean = df_clean.drop_duplicates(subset='user_id', keep='first')
print("final shape:", df_clean.shape)

## Conversion rate per group

In [ ]:
summary = df_clean.groupby('group')['converted'].agg(['count', 'sum', 'mean'])
summary

## Two-proportion z-test

We're testing whether the new page's conversion rate is actually different from the
old page's, or whether the small gap we see is just random noise.

- $x_1, n_1$ = conversions and users in control
- $x_2, n_2$ = conversions and users in treatment

**Null hypothesis (H0):** the two pages convert at the same rate.
**Alternative (H1):** they don't.

In [ ]:
x1, n1 = summary.loc['control', ['sum', 'count']]
x2, n2 = summary.loc['treatment', ['sum', 'count']]

p1 = x1 / n1   # control conversion rate
p2 = x2 / n2   # treatment conversion rate

**Pooled rate** — combine both groups into one, as if H0 is true and there's really
only one shared conversion rate.

$$
\hat{p} = \frac{x_1 + x_2}{n_1 + n_2}
$$

In [ ]:
p_pooled = (x1 + x2) / (n1 + n2)
p_pooled

**Standard error** — how much the difference between the two rates would naturally
bounce around from random sampling alone, if H0 is true.

$$
SE = \sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_1}+\frac{1}{n_2}\right)}
$$

In [ ]:
SE = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
SE

**Z-score** — the difference we actually observed, measured in units of standard error.
The further from 0, the less likely it's just random luck.

$$
Z = \frac{p_2 - p_1}{SE}
$$

In [ ]:
diff = p2 - p1
Z = diff / SE
Z

**P-value** — if H0 were true, what's the probability of seeing a gap this big
(in either direction) just by chance?

$$
p = 2\left(1 - \Phi(|Z|)\right)
$$

In [ ]:
p_value = 2 * (1 - stats.norm.cdf(abs(Z)))

print(f"control rate:   {p1:.4f}")
print(f"treatment rate: {p2:.4f}")
print(f"z-score:        {Z:.4f}")
print(f"p-value:        {p_value:.4f}")

## Conclusion

p-value ≈ 0.19, well above the usual 0.05 cutoff → **not statistically significant.**
The gap between old and new page conversion rates is small enough that it's likely just
random noise, not real evidence one page beats the other.